In [1]:
# === Generic, model-agnostic evaluator for cohort CSVs ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# ----- Paths -----
BASE = "../data/NSW/processed"
COHORT_DIR  = Path("../data/NSW/processed/xgb")
OUT_DIR     = Path("../data/NSW/processed/xgb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
# ----- Metrics -----
def mae(y, yp):
    return np.nanmean(np.abs(yp - y))

def rmse(y, yp):
    e = yp - y
    return np.sqrt(np.nanmean(e * e))

def mape(y, yp):
    denom = np.where(y == 0, np.nan, np.abs(y))
    return np.nanmean(np.abs(yp - y) / denom) * 100

def bias(y, yp):
    return np.nanmean(yp - y)

def metric_table(y, yp):
    return pd.DataFrame([{
        "MAE_MW": mae(y, yp),
        "RMSE_MW": rmse(y, yp),
        "MAPE_%": mape(y, yp),
        "Bias_MW": bias(y, yp)
    }])

In [4]:
# ----- Loading & slot index -----
def load_cohort(csv: Path, pred_col: str) -> pd.DataFrame:
    df = pd.read_csv(csv, parse_dates=["timestamp"])

    if pred_col not in df.columns:
        raise ValueError(f"Column '{pred_col}' not found in {csv.name}")

    # slot index already in XGBoost output file
    
    return df[["timestamp", "slot", "true", pred_col]].rename(columns={pred_col: "yhat"})

In [5]:
# ----- By-slot aggregation -----
def by_hour_metrics(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for slot, g in df.groupby("slot"):
        y, yp = g["true"].to_numpy(), g["yhat"].to_numpy()
        rows.append({
            "slot": int(slot),
            "MAE_MW": mae(y, yp),
            "RMSE_MW": rmse(y, yp),
            "MAPE_%": mape(y, yp),
            "Bias_MW": bias(y, yp),
            "n": len(g)
        })
    return pd.DataFrame(rows).sort_values("slot").reset_index(drop=True)

In [6]:
# ----- Plots -----
def plot_by_slot(per_h: pd.DataFrame, title: str, out_png: Path):
    fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    s = per_h["slot"].to_numpy()
    axs = axs.ravel()

    axs[0].plot(s, per_h["MAE_MW"]);   axs[0].set_title("MAE by slot (MW)");   axs[0].set_ylabel("MAE")
    axs[1].plot(s, per_h["RMSE_MW"]);  axs[1].set_title("RMSE by slot (MW)")
    axs[2].plot(s, per_h["MAPE_%"]);   axs[2].set_title("MAPE by slot (%)");   axs[2].set_ylabel("MAPE (%)")
    axs[3].plot(s, per_h["Bias_MW"]);  axs[3].set_title("Bias by slot (pred − true, MW)"); axs[3].set_xlabel("Hour slot (0–47)")

    for ax in axs: ax.grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180, bbox_inches="tight"); plt.close(fig)

def plot_by_slot_compare(per_h_a: pd.DataFrame, label_a: str,
                         per_h_b: pd.DataFrame, label_b: str,
                         title: str, out_png: Path):

    # Expect columns: ["slot","MAE_MW","RMSE_MW","MAPE_%","Bias_MW"]
    fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    axs = axs.ravel()
    s_a = per_h_a["slot"].to_numpy()
    s_b = per_h_b["slot"].to_numpy()

    def draw(ax, col, yl):
        ax.plot(s_a, per_h_a[col].to_numpy(), label=label_a, lw=1.6)
        ax.plot(s_b, per_h_b[col].to_numpy(), label=label_b, lw=1.6)
        ax.set_ylabel(yl); ax.grid(True, alpha=0.3); ax.legend()

    draw(axs[0], "MAE_MW",  "MAE (MW)")
    axs[0].set_title("MAE by Horizon (MW)")
    draw(axs[1], "RMSE_MW", "RMSE (MW)")
    axs[1].set_title("RMSE by Horizon (MW)")
    draw(axs[2], "MAPE_%",  "MAPE (%)")
    axs[2].set_title("MAPE by Horizon (%)")
    draw(axs[3], "Bias_MW", "Bias (MW)")
    axs[3].set_title("Bias by Horizon (pred − true, MW)")
    axs[3].set_xlabel("Horizon (slots ahead)")
    for ax in axs[:3]: ax.set_xlabel("")  # top row no x label

    fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)

def plot_error_hist(df: pd.DataFrame, title: str, out_png: Path):
    e = (df["yhat"] - df["true"]).to_numpy()
    plt.figure(figsize=(8,5))
    plt.hist(e, bins=60, alpha=0.85)
    plt.title(title); plt.xlabel("Signed error (MW)"); plt.ylabel("Count"); plt.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out_png, dpi=180, bbox_inches="tight"); plt.close()

In [7]:
# ----- One-shot evaluation for a cohort file -----
def evaluate_cohort(name: str, csv: Path, pred_col: str = "op_24h_latest"):
    print(f"[info] Evaluating {pred_col} on {name}")
    df = load_cohort(csv, pred_col)

    # Overall metrics (all rows)
    overall_tbl = metric_table(df["true"].to_numpy(), df["yhat"].to_numpy())
    overall_tbl.index = [pred_col]
    overall_csv = OUT_DIR / f"{pred_col}_{name}_summary.csv"
    overall_tbl.to_csv(overall_csv)
    print(f"[save] summary -> {overall_csv}")

    # By-hour table & plots
    per_h = by_hour_metrics(df)
    per_h_csv = OUT_DIR / f"{pred_col}_{name}_by_slot.csv"
    per_h.to_csv(per_h_csv, index=False)
    print(f"[save] by-hour table -> {per_h_csv}")

    curves_png = OUT_DIR / f"{pred_col}_{name}_by_slot_curves.png"
    plot_by_slot(per_h, f"{pred_col} — {name}", curves_png)
    print(f"[save] curves -> {curves_png}")

    # Error distribution (nice for “how skewed?” in intro)
    hist_png = OUT_DIR / f"{pred_col}_{name}_error_hist.png"
    plot_error_hist(df, f"{pred_col} — error distribution ({name})", hist_png)
    print(f"[save] error hist -> {hist_png}\n")

    return overall_tbl, per_h

In [11]:
# ===== Run for both cohorts with op_latest =====
overall_csv = COHORT_DIR / "cohort_overall_all_xgb_multiout.csv"
hotday_csv  = COHORT_DIR / "cohort_hotday_all_xgb_multiout.csv"

In [15]:
overall_sum, overall_byh = evaluate_cohort("overall_all", overall_csv, pred_col="xgb_pred") 

[info] Evaluating xgb_pred on overall_all
[save] summary -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_summary.csv
[save] by-hour table -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_by_slot.csv
[save] curves -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_by_slot_curves.png
[save] error hist -> ..\data\NSW\processed\xgb\xgb_pred_overall_all_error_hist.png



In [16]:
hotday_sum,  hotday_byh  = evaluate_cohort("hotday_all",  hotday_csv,  pred_col="xgb_pred")

[info] Evaluating xgb_pred on hotday_all
[save] summary -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_summary.csv
[save] by-hour table -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_by_slot.csv
[save] curves -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_by_slot_curves.png
[save] error hist -> ..\data\NSW\processed\xgb\xgb_pred_hotday_all_error_hist.png



In [19]:
# Combined 2-row summary
combined = pd.concat({
    "overall_all": overall_sum.iloc[0],
    "hotday_all":  hotday_sum.iloc[0],
}, axis=1).T.reset_index().rename(columns={"index": "cohort"})
combined_out = OUT_DIR / "xgboost_summary_both.csv"
combined.to_csv(combined_out, index=False)
print(f"[save] combined summary -> {combined_out}")

display(combined)

compare_png = OUT_DIR / "xgboost_overall_vs_hotday_curves.png"
plot_by_slot_compare(
    overall_byh, "overall_all",
    hotday_byh,  "hotday_all",
    title="XGBoost forecast — overall vs hotday",
    out_png=compare_png,
)
print(f"[save] combined curves -> {compare_png}")

[save] combined summary -> ..\data\NSW\processed\xgb\xgboost_summary_both.csv


,cohort,MAE_MW,RMSE_MW,MAPE_%,Bias_MW
0,overall_all,360.279758,551.919387,4.654135,62.242401
1,hotday_all,584.346849,891.455858,6.276731,-330.773173


[save] combined curves -> ..\data\NSW\processed\xgb\xgboost_overall_vs_hotday_curves.png
